# GRU-based Sentiment Classifier Training
Trains a GRU model using Naver Shopping dataset to predict review sentiments (positive/negative).

In [ ]:
import urllib.request
from urllib.request import urlretrieve
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Embedding, Dense, GRU
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import pickle

# Make sure 'eunjeon' package is installed before importing
from eunjeon import Mecab

In [ ]:
# Download Dataset
urlretrieve("https://raw.githubusercontent.com/bab2min/corpus/master/sentiment/naver_shopping.txt", filename="ratings_total.txt")
print("Dataset downloaded successfully.")

In [ ]:
# Load Data
total_data = pd.read_table('ratings_total.txt', names=['ratings', 'reviews'])
print('전체 리뷰 개수 :', len(total_data))

In [ ]:
# Labeling: ratings > 3 is Positive (1), else Negative (0)
total_data['label'] = np.select([total_data.ratings > 3], [1], default=0)
total_data.head()

In [ ]:
# Drop Duplicates
total_data.drop_duplicates(subset=['reviews'], inplace=True)
print('총 샘플의 수 :', len(total_data))

In [ ]:
# Train-Test Split
train_data, test_data = train_test_split(total_data, test_size=0.25, random_state=42)
print('훈련용 리뷰의 개수 :', len(train_data))
print('테스트용 리뷰의 개수 :', len(test_data))

In [ ]:
# Text Preprocessing Function
def preprocess_text(df):
    df['reviews'] = df['reviews'].str.replace(r"[^ㄱ-ㅎㅏ-ㅣ가-힣 ]", "", regex=True)
    df['reviews'] = df['reviews'].str.strip()
    df['reviews'].replace('', np.nan, inplace=True)
    df.dropna(subset=['reviews'], inplace=True)
    return df

train_data = preprocess_text(train_data)
test_data = preprocess_text(test_data)
print('전처리 후 훈련용 샘플의 개수 :', len(train_data))
print('전처리 후 테스트용 샘플의 개수 :', len(test_data))

In [ ]:
# Tokenization Setup
mecab = Mecab()
stopwords = ['도', '는', '다', '의', '가', '이', '은', '한', '에', '하', '고', '을', '를', '인', '듯', '과', '와', '네', '들', '듯', '지', '임', '게']

train_data['tokenized'] = train_data['reviews'].apply(mecab.morphs).apply(lambda x: [w for w in x if w not in stopwords])
test_data['tokenized'] = test_data['reviews'].apply(mecab.morphs).apply(lambda x: [w for w in x if w not in stopwords])

In [ ]:
# Word Frequency Analysis
negative_words = np.hstack(train_data[train_data.label == 0]['tokenized'].values)
positive_words = np.hstack(train_data[train_data.label == 1]['tokenized'].values)

negative_word_count = Counter(negative_words)
print("부정 리뷰 빈출 상위 20개:\n", negative_word_count.most_common(20))

positive_word_count = Counter(positive_words)
print("\n긍정 리뷰 빈출 상위 20개:\n", positive_word_count.most_common(20))

In [ ]:
X_train = train_data['tokenized'].values
y_train = train_data['label'].values
X_test = test_data['tokenized'].values
y_test = test_data['label'].values

In [ ]:
# Tokenizer Fit
tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train)

In [ ]:
# Rare words analysis
threshold = 2
total_cnt = len(tokenizer.word_index)
rare_cnt = sum(1 for v in tokenizer.word_counts.values() if v < threshold)
total_freq = sum(tokenizer.word_counts.values())
rare_freq = sum(v for v in tokenizer.word_counts.values() if v < threshold)

print('단어 집합(vocabulary)의 크기 :', total_cnt)
print(f'등장 빈도가 {threshold - 1}번 이하인 희귀 단어의 수: {rare_cnt}')
print("단어 집합에서 희귀 단어의 비율:", (rare_cnt / total_cnt)*100)
print("전체 등장 빈도에서 희귀 단어 등장 빈도 비율:", (rare_freq / total_freq)*100)

In [ ]:
# Define vocab_size
vocab_size = total_cnt - rare_cnt + 2
print('최종 단어 집합 크기 :', vocab_size)

tokenizer = Tokenizer(vocab_size, oov_token = 'OOV')
tokenizer.fit_on_texts(X_train)
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

In [ ]:
# Max sentence length check
print('리뷰의 최대 길이 :', max(len(l) for l in X_train_seq))
print('리뷰의 평균 길이 :', sum(map(len, X_train_seq))/len(X_train_seq))

In [ ]:
# Padding to max_len
max_len = 80
X_train_pad = pad_sequences(X_train_seq, maxlen = max_len)
X_test_pad = pad_sequences(X_test_seq, maxlen = max_len)

In [ ]:
# Build & Train GRU Model
model = Sequential([
    Embedding(vocab_size, 100),
    GRU(128),
    Dense(1, activation='sigmoid')
])

es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=4)
mc = ModelCheckpoint('best_model.h5', monitor='val_acc', mode='max', verbose=1, save_best_only=True)

model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['acc'])
history = model.fit(X_train_pad, y_train, epochs=15, callbacks=[es, mc], batch_size=60, validation_split=0.2)

In [ ]:
# Evaluate on test set
loaded_model = load_model('best_model.h5')
loss, accuracy = loaded_model.evaluate(X_test_pad, y_test)
print(f"\nTest Accuracy: {accuracy:.4f}")

In [ ]:
# Save Tokenizer
with open('tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)
print("Tokenizer saved successfully.")